# Distillation teacher: B × 2 wide + AutoAugment, 300 epochs

Trains `configs/teacher_B2x_aa_e300.yaml` on **both GPUs** (DataParallel: each GPU gets 64 of the
128 images per batch, so neither GPU sits idle and the session isn't ended early). Runs **silently**: no W&B, no accuracy shown while it runs.
Results appear only in cell 4, after training finishes. See `BRANCH_NOTES.md`, section 6.

Use the **my_env** kernel. Run cells 1–3 in order (wait for each to finish), then leave it.
If the session ends early, open a new session and run cells 1–3 again: cell 2 continues
from the last completed epoch automatically.

## 1. Setup

In [ ]:
import os, re, sys, time, shutil, subprocess
import torch
SCRATCH = os.environ.get('SCRATCH', f"/scratch/{os.environ['USER']}")
os.chdir(os.path.join(SCRATCH, 'pytorch-cifar-resnet'))
os.makedirs('logs', exist_ok=True)
NAME = 'teacher_B2x_aa_e300'
LOG = f'logs/{NAME}.log'
LAST = f'checkpoints/{NAME}/last.pth'
n_gpus = torch.cuda.device_count()
print('torch', torch.__version__, '|', n_gpus, 'GPU(s):', [torch.cuda.get_device_name(i) for i in range(n_gpus)])
assert n_gpus >= 2, 'Expected a 2-GPU session'
assert 'distillation' in open('main.py').read(), 'Old main.py: upload the new one first'

## 2. Start, or continue after a session ended

In [ ]:
if os.path.exists(LOG) and 'Best validation accuracy' in open(LOG, errors='ignore').read():
    print('Already finished: go to cell 4.')
    proc = None
else:
    resume = os.path.exists(LAST)
    cmd = [sys.executable, 'main.py', '--config', f'configs/{NAME}.yaml'] + (['--resume'] if resume else [])
    env = {**os.environ, 'CUDA_VISIBLE_DEVICES': '0,1'}  # both GPUs: DataParallel splits each batch
    proc = subprocess.Popen(cmd, stdout=open(LOG, 'a' if resume else 'w'), stderr=subprocess.STDOUT, env=env)
    START = time.time()
    print(('Continuing from last.pth' if resume else 'Started from scratch'), f'on GPUs 0,1 (pid {proc.pid})')

## 3. Optional: end the session when training finishes

Waits for the run, then ends the session so the GPU isn't held idle. While it waits,
other cells queue behind it (Kernel → Interrupt stops only this cell, not training).

In [ ]:
if proc is not None:
    proc.wait()
    print('training process ended at', time.ctime(), '| exit code', proc.returncode)
    job = os.environ.get('SLURM_JOB_ID')
    if job and shutil.which('scancel'):
        subprocess.run(['scancel', job])

## Status (no accuracy): is it alive, and will it finish in time?

Shows only the epoch count, a time estimate and GPU usage, never accuracy.
Both GPUs should show high utilization and similar memory use.

In [ ]:
text = open(LOG, errors='ignore').read()
done = len(re.findall(r'^Epoch: \d+', text, re.M))
alive = proc is not None and proc.poll() is None
print(f'epoch {done}/300 |', 'running' if alive else 'not running in this session')
m = re.search(r'continuing at epoch (\d+)', text)
first = int(m.group(1)) if m else 0
elapsed = time.time() - START if proc is not None else 0
if alive and done - first > 1:
    per = elapsed / (done - first)
    print(f'~{per:.0f} s/epoch -> ~{per * (300 - done) / 3600:.1f} h to go')
print(subprocess.run(['nvidia-smi', '--query-gpu=index,utilization.gpu,memory.used', '--format=csv'], capture_output=True, text=True).stdout)
job = os.environ.get('SLURM_JOB_ID')
if job:
    print('session time left:', subprocess.run(['squeue', '-j', job, '-h', '-o', '%L'], capture_output=True, text=True).stdout.strip())

## 4. Results (after training finishes; works in any session, CPU too)

Run cell 1 first if this is a new session (on a CPU session, run only its lines up to
`LAST = ...`; the GPU check would stop it).

In [ ]:
text = open(LOG, errors='ignore').read()
val = [(float(l), float(a)) for l, a in re.findall(r'Loss: ([\d.]+) \| Acc: ([\d.]+)% \(\d+/5000\)', text)]
accs = [a for _, a in val]
best = max(accs)
print(f'epochs validated: {len(val)}/300 | finished: {"Best validation accuracy" in text}')
print(f'best val acc {best:.2f}% (epoch {accs.index(best)}) | final val acc {accs[-1]:.2f}% | final val loss {val[-1][0]:.3f}')
print(f'vs. student 97.06%: {best - 97.06:+.2f}%  ->',
      'good teacher' if best >= 97.4 else 'marginal' if best > 97.2 else 'not worth distilling')
print('teacher checkpoint:', f'checkpoints/{NAME}/best.pth', 'exists' if os.path.exists(f'checkpoints/{NAME}/best.pth') else 'MISSING')